In [2]:
from docplex.mp.model import Model
import operator
import pandas as pd
import random
import math
from operator import getitem
import time

# Set up problem parameters
edges = 40
users = 600
services = 4
budget = 3500

# Weights (GLOBAL TRADE-OFF)
w_latency = 0.8
w_availability = 0.2

num_edges = 125
num_users = 816

# --- Data Loading (Assumed available: edgecbd.csv, usercbd.csv) ---
df = pd.read_csv('edgecbd.csv', delimiter=',')
edgeloc = [list(row) for row in df.values]

dfu = pd.read_csv('usercbd.csv', delimiter=',')
userloc = [list(row) for row in dfu.values]

# Edge Detailing
availableservices = []
coverage = []
maxrequests =[]

for i in range(edges):
    coverage.append(random.randint(200,1000)) # in meters
    availableservices.append(random.randint(2, services))
    maxrequests.append(random.randint(50,150))

# Service Detailing
cost = []
latlim = []
# --- NEW: Service Reliability (R_j) ---
# Example reliabilities: Service 1 (eMBB), Service 2, Service 3, Service 4 (uRLLC)
service_reliability = [0.9, 0.99, 0.995, 0.9999, 0.99999] # Higher value means stricter requirement
service_avail = random.choices(service_reliability, k=services)
#print("Selected reliability:", service_avail)

# -------------------------------------

for i in range(services):
    cost.append(random.randint(40,120))
    latlim.append(random.randint(10, 500)) # Latency Limit in ms
    
demand = [] # Demand is an array where demand[k] is the service ID (1-based) requested by user k
for i in range(users):
    demand.append(random.randint(1, services))
    
userArray = []
a = random.sample(range(1, num_users+1), users)
for i in range(users):
    uloc = []
    uloc.append(userloc[a[i-1]-1][0])
    uloc.append(userloc[a[i-1]-1][1])
    userArray.append(uloc)

edgeArray = []
a = random.sample(range(1, num_edges+1), edges)
for i in range(edges):
    eloc = []
    eloc.append(edgeloc[a[i-1]-1][0])
    eloc.append(edgeloc[a[i-1]-1][1])
    edgeArray.append(eloc)

# Calculate haversine distance (in meters)
randist = [] # distance in meters (d*1000)
R = 6371 # km
for e in range(edges):
    distloc = []
    for u in range(users):
        lat1, lon1 = edgeArray[e]
        lat2, lon2 = userArray[u]
        
        dLat = (lat2 - lat1) * (math.pi/180)
        dLon = (lon2 - lon1) * (math.pi/180)
        
        a_val = math.sin(dLat/2)**2 + math.cos(lat1* (math.pi/180)) * math.cos(lat2*(math.pi/180)) * math.sin(dLon/2)**2
        c_val = 2 * math.atan2(math.sqrt(a_val), math.sqrt(1-a_val))
        d_km = R * c_val
        
        distloc.append(d_km * 1000) # distance in meters
    randist.append(distloc)

# Create model
model = Model(name="qos_aware_fault_tolerant_placement")

# Decision variables
x = {(i, j): model.binary_var(name=f"x_{i}_{j}") for i in range(edges) for j in range(services)}
y_primary = {(i, k): model.binary_var(name=f"y_primary_{i}_{k}") for i in range(edges) for k in range(users)}
y_backup = {(i, k): model.binary_var(name=f"y_backup_{i}_{k}") for i in range(edges) for k in range(users)}

# --- OBJECTIVE FUNCTION COMPONENTS ---

# 1. Latency Score (Local QoS aware): Prioritizes primary assignments with large margin under the service's latency limit.
latency_score = model.sum(y_primary[i, k] * (1 - (2 * randist[i][k] / latlim[demand[k] - 1]))
                          for i in range(edges) for k in range(users) if latlim[demand[k] - 1] > 0)


# 2. Availability Score (Local QoS aware): Prioritizes backup assignments for services with HIGH RELIABILITY requirements.
# Factor = 1 / (1 - R_j). Maximizing this factor means prioritizing the highest R_j.
availability_score = model.sum(y_backup[i, k] * (1 / (1 - service_avail[demand[k] - 1]))
                               for i in range(edges) for k in range(users))


# Objective function: Uses GLOBAL weights (w_latency, w_availability) on the LOCAL (service-specific) scores.
model.maximize(w_latency * latency_score + w_availability * availability_score)

# --- CONSTRAINTS (Ensuring Local QoS/Feasibility) ---

# 1. Each user is assigned to at most one primary and one backup server
for k in range(users):
    model.add_constraint(model.sum(y_primary[i, k] for i in range(edges)) <= 1, f"primary_assignment_{k}")
    model.add_constraint(model.sum(y_backup[i, k] for i in range(edges)) <= 1, f"backup_assignment_{k}")

# 2. Primary and backup servers must be distinct
for k in range(users):
    for i in range(edges):
        model.add_constraint(y_primary[i, k] + y_backup[i, k] <= 1, f"distinct_servers_{i}_{k}")

# 3. Backup server can only be assigned if a primary server exists
for k in range(users):
    model.add_constraint(model.sum(y_backup[i, k] for i in range(edges)) <= model.sum(y_primary[i, k] for i in range(edges)),
                         f"backup_only_if_primary_{k}")

# 4. Service availability constraints (Placement must exist)
for i in range(edges):
    for k in range(users):
        model.add_constraint(y_primary[i, k] <= x[i, demand[k] - 1], f"primary_service_availability_{i}_{k}")
        model.add_constraint(y_backup[i, k] <= x[i, demand[k] - 1], f"backup_service_availability_{i}_{k}")

# 5. Latency constraints (Hard QoS requirement check)
for i in range(edges):
    for k in range(users):
        model.add_constraint(2 * randist[i][k] * y_primary[i, k] <= latlim[demand[k] - 1],
                             f"latency_primary_{i}_{k}")
        model.add_constraint(2 * randist[i][k] * y_backup[i, k] <= latlim[demand[k] - 1],
                             f"latency_backup_{i}_{k}")
                             
# 6. Coverage constraints
for i in range(edges):
    for k in range(users):
        model.add_constraint(randist[i][k] * y_primary[i, k] <= coverage[i], f"coverage_primary_{i}_{k}")
        model.add_constraint(randist[i][k] * y_backup[i, k] <= coverage[i], f"coverage_backup_{i}_{k}")

# 7. Budget constraint
model.add_constraint(model.sum(cost[j] * x[i, j] for i in range(edges) for j in range(services)) <= budget, "budget")

# 8. Server capacity constraints
for i in range(edges):
    model.add_constraint(model.sum(x[i, j] for j in range(services)) <= availableservices[i], f"service_type_capacity_{i}")
    model.add_constraint(model.sum(y_primary[i, k] + y_backup[i, k] for k in range(users)) <= maxrequests[i],
                         f"request_capacity_{i}")

# Solve the model
start_time = time.time()
solution = model.solve()
end_time = time.time()
elapsed_time = end_time - start_time
print("Time: ", elapsed_time)

# --- Solution Reporting (same as before) ---
y_p =[]
totalp = 0
y_b =[]
totalb = 0
x_s = []
totals = 0

if solution:
    # Retrieve solution for y_primary
    for i in range(edges):
        for k in range(users):
            if solution.get_value(y_primary[(i, k)]) > 0.5:
                y_p.append(solution.get_value(y_primary[(i, k)]))
                totalp += solution.get_value(y_primary[(i, k)])

    # Retrieve solution for y_backup
    for i in range(edges):
        for k in range(users):
            if solution.get_value(y_backup[(i, k)]) > 0.5:
                y_b.append(solution.get_value(y_backup[(i, k)]))
                totalb += solution.get_value(y_backup[(i, k)])

    # Retrieve solution for x (service placement variables)
    for i in range(edges):
        for j in range(services):
            if solution.get_value(x[(i, j)]) > 0.5:
                x_s.append(solution.get_value(x[(i, j)]))
                totals += solution.get_value(x[(i, j)])

print("Primary: ", (totalp/users)*100)
print("Backup: ", (totalb/users)*100)
print("Services: ", totals)

Time:  0.2889564037322998
Primary:  48.0
Backup:  35.16666666666667
Services:  64.0


In [3]:
from docplex.mp.model import Model
import operator
import pandas as pd
import random
import math
from operator import getitem
import time
import tracemalloc # ADDED

# Set up problem parameters
# ... (rest of your initial parameter setup and data loading) ...
edges = 40
users = 600
services = 4
budget = 3500

# Weights (GLOBAL TRADE-OFF)
w_latency = 0.8
w_availability = 0.2

num_edges = 125
num_users = 816

# --- Data Loading (Assumed available: edgecbd.csv, usercbd.csv) ---
# NOTE: Removed file loading logic as I cannot execute external file I/O.
# Assuming 'edgeloc' and 'userloc' are correctly populated with lists of lists.
# Replacing file load with dummy data for successful execution example:

try:
    df = pd.read_csv('edgecbd.csv', delimiter=',')
    edgeloc = [list(row) for row in df.values]

    dfu = pd.read_csv('usercbd.csv', delimiter=',')
    userloc = [list(row) for row in dfu.values]
except FileNotFoundError:
    print("Warning: Edge/User CSVs not found. Using dummy locations.")
    edgeloc = [[random.uniform(30, 40), random.uniform(-100, -90)] for _ in range(num_edges)]
    userloc = [[random.uniform(30, 40), random.uniform(-100, -90)] for _ in range(num_users)]


# Edge Detailing
availableservices = []
coverage = []
maxrequests =[]
# ... (rest of edge detailing) ...
for i in range(edges):
    coverage.append(random.randint(200,1000)) # in meters
    availableservices.append(random.randint(2, services))
    maxrequests.append(random.randint(50,150))

# Service Detailing
cost = []
latlim = []
service_reliability = [0.9, 0.99, 0.995, 0.9999, 0.99999] # Higher value means stricter requirement
service_avail = random.choices(service_reliability, k=services)
# -------------------------------------

for i in range(services):
    cost.append(random.randint(40,120))
    latlim.append(random.randint(10, 500)) # Latency Limit in ms
    
demand = [] # Demand is an array where demand[k] is the service ID (1-based) requested by user k
for i in range(users):
    demand.append(random.randint(1, services))
    
userArray = []
a = random.sample(range(1, num_users+1), users)
for i in range(users):
    uloc = []
    uloc.append(userloc[a[i-1]-1][0])
    uloc.append(userloc[a[i-1]-1][1])
    userArray.append(uloc)

edgeArray = []
a = random.sample(range(1, num_edges+1), edges)
for i in range(edges):
    eloc = []
    eloc.append(edgeloc[a[i-1]-1][0])
    eloc.append(edgeloc[a[i-1]-1][1])
    edgeArray.append(eloc)

# Calculate haversine distance (in meters)
randist = [] # distance in meters (d*1000)
R = 6371 # km
for e in range(edges):
    distloc = []
    for u in range(users):
        lat1, lon1 = edgeArray[e]
        lat2, lon2 = userArray[u]
        
        dLat = (lat2 - lat1) * (math.pi/180)
        dLon = (lon2 - lon1) * (math.pi/180)
        
        a_val = math.sin(dLat/2)**2 + math.cos(lat1* (math.pi/180)) * math.cos(lat2*(math.pi/180)) * math.sin(dLon/2)**2
        c_val = 2 * math.atan2(math.sqrt(a_val), math.sqrt(1-a_val))
        d_km = R * c_val
        
        distloc.append(d_km * 1000) # distance in meters
    randist.append(distloc)

# ------------------------------------------------------------------
# --- MEMORY TRACING START (Model Building) ---
# ------------------------------------------------------------------
tracemalloc.start()

# Create model
model = Model(name="qos_aware_fault_tolerant_placement")

# Decision variables
x = {(i, j): model.binary_var(name=f"x_{i}_{j}") for i in range(edges) for j in range(services)}
y_primary = {(i, k): model.binary_var(name=f"y_primary_{i}_{k}") for i in range(edges) for k in range(users)}
y_backup = {(i, k): model.binary_var(name=f"y_backup_{i}_{k}") for i in range(edges) for k in range(users)}

# --- OBJECTIVE FUNCTION COMPONENTS ---
# 1. Latency Score (Local QoS aware): Prioritizes primary assignments with large margin under the service's latency limit.
latency_score = model.sum(y_primary[i, k] * (1 - (2 * randist[i][k] / latlim[demand[k] - 1]))
                         for i in range(edges) for k in range(users) if latlim[demand[k] - 1] > 0)


# 2. Availability Score (Local QoS aware): Prioritizes backup assignments for services with HIGH RELIABILITY requirements.
# Factor = 1 / (1 - R_j). Maximizing this factor means prioritizing the highest R_j.
availability_score = model.sum(y_backup[i, k] * (1 / (1 - service_avail[demand[k] - 1]))
                               for i in range(edges) for k in range(users))


# Objective function: Uses GLOBAL weights (w_latency, w_availability) on the LOCAL (service-specific) scores.
model.maximize(w_latency * latency_score + w_availability * availability_score)

# --- CONSTRAINTS (Ensuring Local QoS/Feasibility) ---

# 1. Each user is assigned to at most one primary and one backup server
for k in range(users):
    model.add_constraint(model.sum(y_primary[i, k] for i in range(edges)) <= 1, f"primary_assignment_{k}")
    model.add_constraint(model.sum(y_backup[i, k] for i in range(edges)) <= 1, f"backup_assignment_{k}")

# 2. Primary and backup servers must be distinct
for k in range(users):
    for i in range(edges):
        model.add_constraint(y_primary[i, k] + y_backup[i, k] <= 1, f"distinct_servers_{i}_{k}")

# 3. Backup server can only be assigned if a primary server exists
for k in range(users):
    model.add_constraint(model.sum(y_backup[i, k] for i in range(edges)) <= model.sum(y_primary[i, k] for i in range(edges)),
                          f"backup_only_if_primary_{k}")

# 4. Service availability constraints (Placement must exist)
for i in range(edges):
    for k in range(users):
        model.add_constraint(y_primary[i, k] <= x[i, demand[k] - 1], f"primary_service_availability_{i}_{k}")
        model.add_constraint(y_backup[i, k] <= x[i, demand[k] - 1], f"backup_service_availability_{i}_{k}")

# 5. Latency constraints (Hard QoS requirement check)
for i in range(edges):
    for k in range(users):
        # NOTE: 2 * randist[i][k] is assumed to be Round-Trip Time (RTT) in meters.
        # This constraint assumes latlim is in meters or 2*randist is in ms.
        # If randist is in meters and latlim is in ms, a conversion factor (e.g., speed of light) is needed.
        # Sticking to the original code's expression but adding a note.
        model.add_constraint(2 * randist[i][k] * y_primary[i, k] <= latlim[demand[k] - 1],
                              f"latency_primary_{i}_{k}")
        model.add_constraint(2 * randist[i][k] * y_backup[i, k] <= latlim[demand[k] - 1],
                              f"latency_backup_{i}_{k}")
                             
# 6. Coverage constraints
for i in range(edges):
    for k in range(users):
        model.add_constraint(randist[i][k] * y_primary[i, k] <= coverage[i], f"coverage_primary_{i}_{k}")
        model.add_constraint(randist[i][k] * y_backup[i, k] <= coverage[i], f"coverage_backup_{i}_{k}")

# 7. Budget constraint
model.add_constraint(model.sum(cost[j] * x[i, j] for i in range(edges) for j in range(services)) <= budget, "budget")

# 8. Server capacity constraints
for i in range(edges):
    model.add_constraint(model.sum(x[i, j] for j in range(services)) <= availableservices[i], f"service_type_capacity_{i}")
    model.add_constraint(model.sum(y_primary[i, k] + y_backup[i, k] for k in range(users)) <= maxrequests[i],
                          f"request_capacity_{i}")

# --- MEMORY TRACING END (Model Building) ---
current_mem, peak_mem = tracemalloc.get_traced_memory()
tracemalloc.stop()

# Solve the model
start_time = time.time()
solution = model.solve()
end_time = time.time()
elapsed_time = end_time - start_time
# ------------------------------------------------------------------

print(f"Solver Execution Time: {elapsed_time:.4f} seconds")
print(f"Peak Memory Usage during Model Building: {peak_mem / 10**6:.2f} MB") # NEW METRIC
print("-" * 30)

# --- Solution Reporting (same as before) ---
y_p =[]
totalp = 0
y_b =[]
totalb = 0
x_s = []
totals = 0

if solution:
    # Retrieve solution for y_primary
    for i in range(edges):
        for k in range(users):
            if solution.get_value(y_primary[(i, k)]) > 0.5:
                y_p.append(solution.get_value(y_primary[(i, k)]))
                totalp += solution.get_value(y_primary[(i, k)])

    # Retrieve solution for y_backup
    for i in range(edges):
        for k in range(users):
            if solution.get_value(y_backup[(i, k)]) > 0.5:
                y_b.append(solution.get_value(y_backup[(i, k)]))
                totalb += solution.get_value(y_backup[(i, k)])

    # Retrieve solution for x (service placement variables)
    for i in range(edges):
        for j in range(services):
            if solution.get_value(x[(i, j)]) > 0.5:
                x_s.append(solution.get_value(x[(i, j)]))
                totals += solution.get_value(x[(i, j)])

    print(f"Primary Assignments (Coverage): {(totalp/users)*100:.2f}%")
    print(f"Backup Assignments (Redundancy): {(totalb/users)*100:.2f}%")
    print(f"Total Services Deployed: {totals}")
else:
    print("Model could not be solved. It may be infeasible or time limit reached.")

Solver Execution Time: 2.1726 seconds
Peak Memory Usage during Model Building: 208.40 MB
------------------------------
Primary Assignments (Coverage): 35.00%
Backup Assignments (Redundancy): 26.83%
Total Services Deployed: 47.00000000000226
